# **Advanced RAG Assignment - Part 1**

## Setup

In [21]:
import os
import subprocess
import logging
from pathlib import Path
from dotenv import load_dotenv
from langchain_community.chat_models import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate

# Load environment variables from .env file
load_dotenv()

# Setup logging
logging.basicConfig(
    format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s',
    level=logging.INFO
)
logger = logging.getLogger(__name__)

In [22]:
if os.getenv("GROQ_API_KEY"):
    logger.info("GROQ_API_KEY is set")
else:
    logger.warning("No API key found! Please set GROQ_API_KEY in your .env file")

[2026-03-03 22:52:16,387] p2007 {3541512351.py:2} INFO - GROQ_API_KEY is set


In [23]:
MODEL_ID = "groq/llama-3.3-70b-versatile"

llm = ChatLiteLLM(
    model=MODEL_ID,
    temperature=0
)
logger.info(f"Using model: {MODEL_ID}")

[2026-03-03 22:52:16,418] p2007 {172078051.py:7} INFO - Using model: groq/llama-3.3-70b-versatile


In [24]:
# Path to the cloned mcp-gateway-registry codebase
CODEBASE_PATH = Path('../../mcp-gateway-registry').resolve()
logger.info(f"Codebase path: {CODEBASE_PATH}")
logger.info(f"Exists: {CODEBASE_PATH.exists()}")

[2026-03-03 22:52:16,430] p2007 {342584891.py:3} INFO - Codebase path: /home/ubuntu/dsan6725/assignments/spring-2026-a03-alisonmanna/mcp-gateway-registry
[2026-03-03 22:52:16,431] p2007 {342584891.py:4} INFO - Exists: True


## Step 1: Query Classification

First, a function for classifying the incoming question to figure out what type of information is needed. This is to determine which bash tools to use

In [25]:
def classify_query(question):
    """
    Classify a question into a query type using keyword matching.
    Returns either dependency, entry_point, file_types, api_endpoints, oauth_extension, authentication, general
    """
    q = question.lower()

    if any(kw in q for kw in ['dependency', 'dependencies', 'package', 'requirements', 'install']):
        return 'dependency'
    elif any(kw in q for kw in ['entry point', 'main entry', 'main file', 'entry file']):
        return 'entry_point'
    elif any(kw in q for kw in ['language', 'file type', 'file types', 'programming language', 'typescript', 'dockerfile']):
        return 'file_types'
    elif any(kw in q for kw in ['endpoint', 'api endpoint', 'scope', 'routes']):
        return 'api_endpoints'
    elif 'add support' in q or 'new oauth' in q or 'okta' in q or ('new' in q and 'provider' in q):
        return 'oauth_extension'
    elif any(kw in q for kw in ['auth', 'token', 'oauth', 'authorization', 'flow', 'login']):
        return 'authentication'
    else:
        return 'general'

# Sanity check
test_questions = [
    "What Python dependencies does this project use?",
    "What is the main entry point file for the registry service?",
    "What programming languages and file types are used in this repository?",
    "How does the authentication flow work, from token validation to user authorization?",
    "What are all the API endpoints available in the registry service and what scopes do they require?",
    "How would you add support for a new OAuth provider (e.g., Okta) to the authentication system?",
]

for question in test_questions:
    logger.info(f"{classify_query(question)} <- {question[:60]}")

[2026-03-03 22:52:16,442] p2007 {1670943069.py:34} INFO - dependency <- What Python dependencies does this project use?
[2026-03-03 22:52:16,444] p2007 {1670943069.py:34} INFO - entry_point <- What is the main entry point file for the registry service?
[2026-03-03 22:52:16,445] p2007 {1670943069.py:34} INFO - file_types <- What programming languages and file types are used in this r
[2026-03-03 22:52:16,445] p2007 {1670943069.py:34} INFO - authentication <- How does the authentication flow work, from token validation
[2026-03-03 22:52:16,447] p2007 {1670943069.py:34} INFO - api_endpoints <- What are all the API endpoints available in the registry ser
[2026-03-03 22:52:16,447] p2007 {1670943069.py:34} INFO - oauth_extension <- How would you add support for a new OAuth provider (e.g., Ok


## Step 2: Bash Tool Selection

Now based on the query type, we can use a function to pick the right bash commands to retrieve the relevant context from the codebase.

In [26]:
def get_bash_commands(query_type, codebase_path):
    """
    Return list of bash commands to run based on query type
    """
    p = str(codebase_path)

    if query_type == 'dependency':
        return [
            f"cat {p}/pyproject.toml",
            f"find {p} -name 'requirements*.txt' -not -path '*/node_modules/*' -exec echo '=== {{}} ===' \\; -exec cat {{}} \\;",
            f"find {p} -name 'package.json' -not -path '*/node_modules/*' | head -5",
        ]

    elif query_type == 'entry_point':
        return [
            f"find {p} -name 'main.py' -not -path '*/node_modules/*'",
            f"head -60 {p}/registry/main.py",
            f"head -30 {p}/Dockerfile 2>/dev/null",
        ]

    elif query_type == 'file_types':
        return [
            f"tree {p} -L 2 --dirsfirst 2>/dev/null | head -40",
            f"find {p} -type f -not -path '*/.git/*' -not -path '*/node_modules/*' -not -path '*/__pycache__/*' | sed 's/.*\\.//' | sort | uniq -c | sort -rn | head -25",
        ]

    elif query_type == 'api_endpoints':
        return [
            f"grep -rn '@router\\.' --include='*.py' {p}/registry/api/ | head -80",
            f"cat {p}/auth_server/scopes.yml",
        ]

    elif query_type == 'authentication':
        return [
            f"head -100 {p}/auth_server/server.py",
            f"cat {p}/auth_server/scopes.yml",
            f"grep -rn 'token\\|validate\\|authorize' --include='*.py' {p}/registry/auth/ 2>/dev/null | head -40",
        ]

    elif query_type == 'oauth_extension':
        return [
            f"ls {p}/auth_server/providers/",
            f"cat {p}/auth_server/oauth2_providers.yml",
            f"cat {p}/auth_server/providers/base.py",
            f"cat {p}/auth_server/providers/factory.py",
        ]
    
    # general
    else:
        return [
            f"tree {p} -L 2 --dirsfirst 2>/dev/null | head -40",
            f"head -60 {p}/README.md 2>/dev/null",
        ]

logger.info("get_bash_commands defined")

[2026-03-03 22:52:16,471] p2007 {4039177498.py:55} INFO - get_bash_commands defined


## Step 3: Execute Bash Commands

Next we need a function to run the selected commands using the python `subprocess` module and collect the output as context.

In [27]:
MAX_OUTPUT_CHARS = 8000

def run_bash_commands(commands):
    """
    Execute list of bash commands and return combined output as a string (truncates output if it gets too large)
    """
    all_output = []

    for cmd in commands:
        logger.info(f"Running: {cmd[:80]}")
        try:
            result = subprocess.run(
                cmd, shell=True, capture_output=True, text=True, timeout=30
            )
            output = result.stdout

            if result.returncode != 0 and result.stderr:
                output += f"\nSTDERR: {result.stderr[:200]}"

        except subprocess.TimeoutExpired:
            output = "[Command timed out]"

        all_output.append(f"$ {cmd}\n{output}")

    combined = "\n\n".join(all_output)

    if len(combined) > MAX_OUTPUT_CHARS:
        combined = combined[:MAX_OUTPUT_CHARS] + "\n...[output truncated]"

    return combined


logger.info("run_bash_commands defined")

[2026-03-03 22:52:16,488] p2007 {2394196124.py:33} INFO - run_bash_commands defined


## Step 4: Answer Generation

Passing the retrieved context along with the question to the LLM to generate an answer

In [28]:
system_prompt = (
    "You are a helpful assistant that answers questions about a software codebase. "
    "Use the following context retrieved from bash commands to answer the question. "
    "Always reference specific files when possible. "
    "If you don't know the answer, say so.\n\n"
    "Context from codebase:\n{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{question}"),
])

def code_qa_pipeline(question):
    """
    Full pipeline: classify -> select tools -> execute -> generate answer
    """
    query_type = classify_query(question)
    logger.info(f"Query type: {query_type}")

    commands = get_bash_commands(query_type, CODEBASE_PATH)
    context = run_bash_commands(commands)
    logger.info(f"Context length: {len(context)} chars")

    chain = prompt | llm
    response = chain.invoke({"context": context, "question": question})

    return {
        "question": question,
        "query_type": query_type,
        "commands": commands,
        "answer": response.content,
    }

logger.info("code_qa_pipeline created")

[2026-03-03 22:52:16,505] p2007 {2537826858.py:35} INFO - code_qa_pipeline created


## Run Test Questions

In [29]:
test_questions = [
    "What Python dependencies does this project use?",
    "What is the main entry point file for the registry service?",
    "What programming languages and file types are used in this repository? (e.g., Python, TypeScript, YAML, JSON, Dockerfile, etc.)",
    "How does the authentication flow work, from token validation to user authorization?",
    "What are all the API endpoints available in the registry service and what scopes do they require?",
    "How would you add support for a new OAuth provider (e.g., Okta) to the authentication system? What files would need to be modified and what interfaces must be implemented?",
]

results = []
for i, q in enumerate(test_questions, 1):
    logger.info(f"\n{'='*60}")
    logger.info(f"Question {i}: {q}")
    result = code_qa_pipeline(q)
    results.append(result)
    logger.info(f"Answer preview: {result['answer'][:200]}...")

[2026-03-03 22:52:16,518] p2007 {1444740016.py:12} INFO - 
[2026-03-03 22:52:16,519] p2007 {1444740016.py:13} INFO - Question 1: What Python dependencies does this project use?
[2026-03-03 22:52:16,520] p2007 {2537826858.py:19} INFO - Query type: dependency
[2026-03-03 22:52:16,522] p2007 {2394196124.py:10} INFO - Running: cat /home/ubuntu/dsan6725/assignments/spring-2026-a03-alisonmanna/mcp-gateway-re
[2026-03-03 22:52:16,529] p2007 {2394196124.py:10} INFO - Running: find /home/ubuntu/dsan6725/assignments/spring-2026-a03-alisonmanna/mcp-gateway-r
[2026-03-03 22:52:16,664] p2007 {2394196124.py:10} INFO - Running: find /home/ubuntu/dsan6725/assignments/spring-2026-a03-alisonmanna/mcp-gateway-r
[2026-03-03 22:52:16,673] p2007 {2537826858.py:23} INFO - Context length: 5926 chars
22:52:16 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= llama-3.3-70b-versatile; provider = groq
[2026-03-03 22:52:16,689] p2007 {utils.py:3898} INFO - 
LiteLLM completion() model= llama-3.3-70b-vers

In [30]:
# Show full results
for i, r in enumerate(results, 1):
    print(f"\n{'-'*60}")
    print(f"Q{i} [{r['query_type']}]: {r['question']}")
    print(f"\nCommands used:")
    for cmd in r['commands']:
        print(f"  - {cmd[:80]}")
    print(f"\nAnswer:\n{r['answer']}")


------------------------------------------------------------
Q1 [dependency]: What Python dependencies does this project use?

Commands used:
  - cat /home/ubuntu/dsan6725/assignments/spring-2026-a03-alisonmanna/mcp-gateway-re
  - find /home/ubuntu/dsan6725/assignments/spring-2026-a03-alisonmanna/mcp-gateway-r
  - find /home/ubuntu/dsan6725/assignments/spring-2026-a03-alisonmanna/mcp-gateway-r

Answer:
The project uses the following Python dependencies, as specified in the `pyproject.toml` file:

1. `aiofiles>=24.1.0`
2. `fastapi>=0.115.12`
3. `itsdangerous>=2.2.0`
4. `jinja2>=3.1.6`
5. `mcp>=1.9.3`
6. `pydantic>=2.11.3`
7. `pydantic-settings>=2.0.0`
8. `httpx>=0.27.0`
9. `python-dotenv>=1.1.0`
10. `python-multipart>=0.0.20`
11. `uvicorn[standard]>=0.34.2`
12. `faiss-cpu>=1.7.4`
13. `sentence-transformers>=3.0.0`
14. `websockets>=15.0.1`
15. `scikit-learn>=1.3.0`
16. `torch>=1.6.0`
17. `huggingface-hub>=0.31.1`
18. `bandit>=1.8.3`
19. `langchain-mcp-adapters>=0.0.11`
20. `langgraph>=0

## Save Results

In [31]:
output_path = Path('../../part1_results.txt')

with open(output_path, 'w') as f:
    for i, r in enumerate(results, 1):
        f.write(f"Question {i}: {r['question']}\n")
        f.write(f"Query Type: {r['query_type']}\n")
        f.write(f"Commands Used:\n")
        for cmd in r['commands']:
            f.write(f"  {cmd}\n")
        f.write(f"\nAnswer:\n{r['answer']}\n")
        f.write(f"\n{'='*60}\n\n")

logger.info(f"Saved results to {output_path}")

[2026-03-03 22:52:25,836] p2007 {2399636101.py:13} INFO - Saved results to ../../part1_results.txt
